In [5]:
# Первая ячейка: Импорт библиотек и определение классов
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
import math
from dataclasses import dataclass
from enum import Enum
import os

# Настройка для отображения графиков в Jupyter
%matplotlib inline

class Predecessor(Enum):
    """Типы предшественников"""
    WINTER_WHEAT = "озимая пшеница"
    CORN = "кукуруза"
    SUNFLOWER = "подсолнечник"
    LEGUMES = "бобовые"
    FALLOW = "чистый пар"
    POTATOES = "картофель"


@dataclass
class Soil:
    """Состояние почвы"""
    humus: float  # % (гумус)
    ph: float  # pH (KCl)
    n_mg_kg: float  # минеральный азот (нитраты + аммоний), мг/кг
    p_mg_kg: float  # подвижный фосфор (по Чирикову), мг/кг
    k_mg_kg: float  # обменный калий (по Чирикову), мг/кг
    soil_type: str = "chernozem"

@dataclass
class Weather:
    """Погодные условия вегетационного периода"""
    precipitation_mm: float  # сумма осадков, мм
    avg_temp_c: float  # средняя температура, °C


@dataclass
class PredecessorInfo:
    """Информация о предшественнике"""
    crop: Predecessor
    yield_t_ha: float  # урожайность предшественника, т/га


@dataclass
class FertilizerRates:
    """Результат: рекомендуемые дозы удобрений"""
    n_kg_ha: float  # азот (N) кг/га д.в.
    p_kg_ha: float  # фосфор (P2O5) кг/га д.в.
    k_kg_ha: float  # калий (K2O) кг/га д.в.

    def __str__(self):
        return (f"Рекомендуемые дозы удобрений:\n"
                f"  N (азот) : {self.n_kg_ha:.1f} кг/га\n"
                f"  P (фосфор): {self.p_kg_ha:.1f} кг/га\n"
                f"  K (калий) : {self.k_kg_ha:.1f} кг/га")


def calculate_fertilizer_rates(
        target_crop: str,
        target_yield_t_ha: float,
        soil: Soil,
        weather: Weather,
        predecessor: PredecessorInfo
) -> FertilizerRates:
    # ----- 1. Базовый вынос на планируемый урожай -----
    # Вынос на 1 т основной продукции (кг N/P2O5/K2O)
    # Данные для пшеницы
    if target_crop == "wheat":
        uptake_per_ton = {"n": 30.0, "p": 12.0, "k": 25.0}  # кг/т
    else:
        # по умолчанию для других
        uptake_per_ton = {"n": 25.0, "p": 10.0, "k": 20.0}

    total_uptake = {
        "n": uptake_per_ton["n"] * target_yield_t_ha,
        "p": uptake_per_ton["p"] * target_yield_t_ha,
        "k": uptake_per_ton["k"] * target_yield_t_ha,
    }

    # ----- 2. Доступность из почвы -----
    # Азот: доступность из почвы (коэффициент использования N из гумуса)
    humus_factor = min(1.2, max(0.6, 1.0 - (soil.humus - 4.0) * 0.05))

    # Фосфор: коэффициент доступности из почвы зависит от pH и содержания P
    if soil.p_mg_kg > 100:
        p_soil_supply = 0.15
    elif soil.p_mg_kg > 60:
        p_soil_supply = 0.25
    elif soil.p_mg_kg > 30:
        p_soil_supply = 0.45
    else:
        p_soil_supply = 0.65

    # Коррекция по pH (при высоком pH доступность P снижается)
    if soil.ph > 7.5:
        p_soil_supply *= 0.7
    elif soil.ph < 5.5:
        p_soil_supply *= 0.8

    # Калий: коэффициент доступности из почвы
    if soil.k_mg_kg > 200:
        k_soil_supply = 0.15
    elif soil.k_mg_kg > 120:
        k_soil_supply = 0.30
    elif soil.k_mg_kg > 70:
        k_soil_supply = 0.50
    else:
        k_soil_supply = 0.70

    # Запас элемента в почве (пересчет в кг/га, слой 0-30 см)
    soil_supply_kg = {
        "n": soil.n_mg_kg * 3.0,
        "p": soil.p_mg_kg * 3.0,
        "k": soil.k_mg_kg * 3.0,
    }

    # Доступное из почвы количество
    available_from_soil = {
        "n": soil_supply_kg["n"] * humus_factor,
        "p": soil_supply_kg["p"] * p_soil_supply,
        "k": soil_supply_kg["k"] * k_soil_supply,
    }

    # ----- 3. Поправка на погоду -----
    # Азот: осадки и температура
    precip_factor = 1.0
    if weather.precipitation_mm < 200:  # засуха
        precip_factor = 0.7
    elif weather.precipitation_mm > 450:  # избыток осадков
        precip_factor = 1.25

    temp_factor = 1.0
    if weather.avg_temp_c < 12:  # холодно
        temp_factor = 0.85
    elif weather.avg_temp_c > 22:
        temp_factor = 1.1

    weather_factor_n = precip_factor * temp_factor

    # Для фосфора и калия влияние погоды слабее
    weather_factor_p = 1.0
    weather_factor_k = 1.0
    if weather.avg_temp_c < 10:
        weather_factor_p = 1.15  # холодная весна - хуже доступность P
        weather_factor_k = 0.95

    # ----- 4. Поправка на предшественник (только для азота) -----
    predecessor_factor = 1.0
    predecessor_n_credit = 0.0

    if predecessor.crop == Predecessor.LEGUMES:
        predecessor_factor = 0.65
        # Бобовые оставляют 40-80 кг/га азота
        predecessor_n_credit = 50.0
    elif predecessor.crop == Predecessor.FALLOW:
        predecessor_factor = 0.70
    elif predecessor.crop == Predecessor.WINTER_WHEAT:
        predecessor_factor = 1.05
    elif predecessor.crop == Predecessor.CORN:
        predecessor_factor = 1.00
    elif predecessor.crop == Predecessor.SUNFLOWER:
        predecessor_factor = 1.10  # сильно истощает
    elif predecessor.crop == Predecessor.POTATOES:
        predecessor_factor = 0.95

    # Учет урожайности предшественника (высокий урожай -> меньше остаточного питания)
    if predecessor.yield_t_ha > 5.0:
        predecessor_factor *= 1.05

    # ----- 5. Коэффициент использования из удобрений -----
    # Доля, которую растение реально усвоит из внесенного удобрения
    fertilizer_efficiency = {
        "n": 0.65,
        "p": 0.35,  # фосфор усваивается хуже
        "k": 0.55,
    }

    # ----- 6. Итоговый расчет доз -----
    # Необходимо внести = (Вынос - Доступно из почвы) / КПД удобрений
    # с учетом поправок

    # Азот (с учетом погоды и предшественника)
    required_n = max(0, total_uptake["n"] - available_from_soil["n"] - predecessor_n_credit)
    required_n *= predecessor_factor
    required_n *= weather_factor_n
    dose_n = required_n / fertilizer_efficiency["n"]

    # Фосфор
    required_p = max(0, total_uptake["p"] - available_from_soil["p"])
    required_p *= weather_factor_p
    dose_p = required_p / fertilizer_efficiency["p"]

    # Калий
    required_k = max(0, total_uptake["k"] - available_from_soil["k"])
    required_k *= weather_factor_k
    dose_k = required_k / fertilizer_efficiency["k"]

    # Округление и минимальные стартовые дозы
    dose_n = max(30.0, round(dose_n / 5) * 5)
    dose_p = max(15.0, round(dose_p / 5) * 5)
    dose_k = max(0.0, round(dose_k / 5) * 5)  # калий может не требоваться

    return FertilizerRates(n_kg_ha=dose_n, p_kg_ha=dose_p, k_kg_ha=dose_k)


def predict_base_yield(soil: Soil, weather: Weather) -> float:
    """
    Прогноз базовой урожайности (без удобрений)
    на основе почвенных и погодных условий.
    """
    # Базовый потенциал от гумуса (2 т/га при 2% гумуса)
    base = 2.0 + (soil.humus - 2.0) * 0.4

    # Поправка на pH
    if soil.ph < 5.5 or soil.ph > 8.0:
        base *= 0.8
    elif soil.ph < 6.0:
        base *= 0.95

    # Поправка на содержание элементов
    base += min(2.0, soil.n_mg_kg / 40)
    base += min(1.5, soil.p_mg_kg / 100)
    base += min(1.0, soil.k_mg_kg / 200)

    # Поправка на погоду
    if weather.precipitation_mm < 200:
        base *= 0.65
    elif weather.precipitation_mm > 500:
        base *= 0.85

    if weather.avg_temp_c < 10 or weather.avg_temp_c > 30:
        base *= 0.85

    return max(1.0, min(7.0, round(base, 1)))


def predict_yield_with_fertilizer(base_yield: float, rates: FertilizerRates, target_yield: float) -> float:
    """
    Прогноз урожайности при внесении рекомендованных доз удобрений.
    """
    # Эффективность удобрений (упрощенно)
    n_effect = min(2.5, rates.n_kg_ha / 80)  # максимум +2.5 т/га
    p_effect = min(1.0, rates.p_kg_ha / 100)  # максимум +1.0 т/га
    k_effect = min(0.8, rates.k_kg_ha / 120)  # максимум +0.8 т/га

    total_effect = n_effect + p_effect + k_effect

    predicted = base_yield + total_effect
    # Не может быть намного выше плановой урожайности
    predicted = min(target_yield + 0.8, predicted)

    return round(predicted, 1)

In [8]:
# Вторая ячейка: Создание примерных файлов CSV (если они не существуют)
# Создаем примерные данные, если файлы отсутствуют
if not os.path.exists('soil.csv'):
    soil_data = pd.DataFrame({
        'humus': [3.5, 4.2, 3.8, 4.5, 4.0],
        'ph': [6.2, 6.5, 6.8, 6.3, 6.6],
        'n_mg_kg': [35, 42, 38, 45, 40],
        'p_mg_kg': [55, 62, 58, 65, 60],
        'k_mg_kg': [110, 125, 115, 130, 120],
        'soil_type': ['chernozem', 'chernozem', 'chernozem', 'chernozem', 'chernozem']
    })
    soil_data.to_csv('soil.csv', index=False)
    print("Создан файл soil.csv с примерными данными")
else:
    print("Файл soil.csv уже существует")

if not os.path.exists('weather.csv'):
    weather_data = pd.DataFrame({
        'precipitation_mm': [320, 350, 340, 360, 345],
        'avg_temp_c': [17.5, 18.0, 17.8, 18.2, 17.9]
    })
    weather_data.to_csv('weather.csv', index=False)
    print("Создан файл weather.csv с примерными данными")
else:
    print("Файл weather.csv уже существует")

Файл soil.csv уже существует
Файл weather.csv уже существует


In [7]:
# Третья ячейка: Загрузка данных и расчет
# Загрузка CSV
def load_CSV():
    data_Soil = pd.read_csv('soil.csv')
    data_Weather = pd.read_csv('weather.csv')
    index_Soil = 1  # используем вторую строку (индекс 1)
    index_Weather = 1
    
    soil = Soil(
        humus=data_Soil["humus"][index_Soil],
        ph=data_Soil["ph"][index_Soil],
        n_mg_kg=data_Soil["n_mg_kg"][index_Soil],
        p_mg_kg=data_Soil["p_mg_kg"][index_Soil],
        k_mg_kg=data_Soil["k_mg_kg"][index_Soil],
        soil_type=data_Soil["soil_type"][index_Soil]
    )
    weather = Weather(
        precipitation_mm=data_Weather["precipitation_mm"][index_Weather],
        avg_temp_c=data_Weather["avg_temp_c"][index_Weather]
    )
    predecessor = PredecessorInfo(
        crop=Predecessor.WINTER_WHEAT,
        yield_t_ha=4.2
    )
    return soil, weather, predecessor

# Загружаем данные
soil, weather, predecessor = load_CSV()

print("=== ЗАГРУЖЕННЫЕ ДАННЫЕ ===")
print(f"Почва: гумус={soil.humus}%, pH={soil.ph}, N={soil.n_mg_kg} мг/кг, P={soil.p_mg_kg} мг/кг, K={soil.k_mg_kg} мг/кг")
print(f"Погода: осадки={weather.precipitation_mm} мм, температура={weather.avg_temp_c}°C")
print(f"Предшественник: {predecessor.crop.value}, урожайность={predecessor.yield_t_ha} т/га")


=== ЗАГРУЖЕННЫЕ ДАННЫЕ ===
Почва: гумус=4.2%, pH=6.5, N=42 мг/кг, P=62 мг/кг, K=125 мг/кг
Погода: осадки=350 мм, температура=18.0°C
Предшественник: озимая пшеница, урожайность=4.2 т/га


In [9]:
# Четвертая ячейка: Расчет удобрений
target_yield = 6.0

# Расчет для пшеницы с планируемой урожайностью
rates = calculate_fertilizer_rates(
    target_crop="wheat",
    target_yield_t_ha=target_yield,
    soil=soil,
    weather=weather,
    predecessor=predecessor
)

print(rates)
print("\n--- Детали расчета ---")
print(f"Вынос NPK для пшеницы на {target_yield} т/га: N≈{30*target_yield:.0f}, P≈{12*target_yield:.0f}, K≈{25*target_yield:.0f} кг/га")

Рекомендуемые дозы удобрений:
  N (азот) : 90.0 кг/га
  P (фосфор): 75.0 кг/га
  K (калий) : 70.0 кг/га

--- Детали расчета ---
Вынос NPK для пшеницы на 6.0 т/га: N≈180, P≈72, K≈150 кг/га
